# 🎯 CV Matcher - LoRA Fine-Tuning with Gemma-2B

Bu notebook, **Gemma-2B** modelini CV parsing ve CV-JD matching için **LoRA** ile fine-tune eder.

## Kullanım
1. **Runtime → Change runtime type** → **T4 GPU** seç
2. **Ctrl+F9** ile tüm hücreleri çalıştır
3. Fine-tuning bittiğinde adapter'lar Google Drive'a kaydedilir

---

## 📦 Adım 1: GPU Kontrolü ve Kütüphaneler

In [ ]:
# GPU kontrolü
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Kütüphaneleri yükle
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate peft trl datasets bitsandbytes scipy sentencepiece

## 📂 Adım 2: Projeyi ve Dataset'i Yükle

In [ ]:
# GitHub'dan projeyi çek
!git clone https://github.com/ruveydagundogan/cvmatcher.git
%cd cvmatcher

# Dataset'i kontrol et
import json
with open("backend/finetune/data/cv_parse_dataset.json") as f:
    cv_data = json.load(f)
with open("backend/finetune/data/cv_jd_match_dataset.json") as f:
    match_data = json.load(f)
print(f"CV Parse: {len(cv_data)} örnek")
print(f"CV-JD Match: {len(match_data)} örnek")

## 🧠 Adım 3: CV Parsing Modeli Fine-Tune

Bu adımda Gemma-2B'yi CV'lerden skill/experience/education çıkarması için eğitiyoruz.

**Hiperparametreler:**
- LoRA rank (r): 8 (düşük = hızlı, yeterli)
- LoRA alpha: 16
- Epoch: 5 (küçük dataset için ideal)
- Batch size: 4
- 4-bit quantization: True (VRAM yetmezse)

In [ ]:
import sys
sys.path.append("backend/finetune")

# CV Parse modelini fine-tune et
!python backend/finetune/train_lora.py \
    --base-model google/gemma-2b-it \
    --data backend/finetune/data/cv_parse_dataset.json \
    --output-dir /content/drive/MyDrive/cvmatcher-lora/cv-parser-v1 \
    --mode cv-parse \
    --epochs 5 \
    --batch-size 4 \
    --max-length 512 \
    --lr 2e-4 \
    --quantize

## 🧠 Adım 4: CV-JD Matching Modeli Fine-Tune

Şimdi de CV-JD eşleştirme modelini eğitiyoruz.

In [ ]:
!python backend/finetune/train_lora.py \
    --base-model google/gemma-2b-it \
    --data backend/finetune/data/cv_jd_match_dataset.json \
    --output-dir /content/drive/MyDrive/cvmatcher-lora/cv-jd-matcher-v1 \
    --mode cv-jd-match \
    --epochs 5 \
    --batch-size 4 \
    --max-length 512 \
    --lr 2e-4 \
    --quantize

## 🧪 Adım 5: Fine-Tune Öncesi vs Sonrası Test

Base model ve fine-tuned model arasındaki farkı görelim.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel

test_cv = """Python backend developer with 4 years experience. 
Skilled in Django, FastAPI, PostgreSQL, Redis, Celery, Docker.
Built REST APIs serving 50K requests per minute.
Bachelor's in Software Engineering."""

prompt = f"""<start_of_turn>user
Parse the following CV text and extract structured information: skills, experience, education, and a brief summary.

{test_cv}
<end_of_turn>
<start_of_turn>model
"""

print("=" * 60)
print("BASE MODEL (Gemma-2B) TESTİ")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = base_model.generate(**inputs, max_new_tokens=256, temperature=0.1)
base_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(base_response[-500:] if len(base_response) > 500 else base_response)

In [ ]:
print("=" * 60)
print("FINE-TUNED MODEL (LoRA) TESTİ")
print("=" * 60)

finetuned = PeftModel.from_pretrained(
    base_model, "/content/drive/MyDrive/cvmatcher-lora/cv-parser-v1"
)
finetuned.eval()

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = finetuned.generate(**inputs, max_new_tokens=256, temperature=0.1)
ft_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(ft_response[-500:] if len(ft_response) > 500 else ft_response)

## 📥 Adım 6: Adapter'ları İndir

Google Drive'a kaydedilen adapter'ları bilgisayarına indir.
Drive'dan → sağ tık → İndir

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Adapter'lar Drive'a kaydedildi:")
!ls -la /content/drive/MyDrive/cvmatcher-lora/ 2>/dev/null || echo "(Drive'a yazma hatası, local'e kaydediliyor...)"

# Alternatif: Zip olarak indir
!zip -r /content/cvmatcher-lora.zip /content/drive/MyDrive/cvmatcher-lora/ 2>/dev/null
from google.colab import files
files.download('/content/cvmatcher-lora.zip')

## 📋 Sonuç

Fine-tuning tamamlandı! Şimdi adapter'ları Mac'indeki Ollama'ya yükleyebilirsin.

### Sonraki adımlar:
1. İndirdiğin `cvmatcher-lora.zip` dosyasını aç
2. `cv-parser-v1` ve `cv-jd-matcher-v1` klasörlerini `backend/finetune/adapters/` altına koy
3. Ollama'da custom model oluştur:
   ```bash
   ollama create cv-parser -f backend/finetune/adapters/cv-parser-v1/Modelfile
   ollama create cv-jd-matcher -f backend/finetune/adapters/cv-jd-matcher-v1/Modelfile
   ```
4. Backend'de model adını güncelle ve test et